# Introduction to Statistics & Scientific Python Tooling
---
> This notebook will help you master core statistical principles and scientific Python libraries including NumPy, SciPy, Scikit-learn, and StatsModels. It breaks down critical mechanics across hypothesis testing, z-score transformations, polynomial expansions, confidence intervals, and outlier detection with robust production code.

Topics Covered:
- Cover NumPy PRNG, statistical aggregation, and array transformations
- Audit SciPy hypothesis testing mechanics and mode extraction
- Detail Scikit-Learn standard scaling and polynomial feature engineering
- Implement StatsModels weighted descriptive statistics and confidence bounds


### 1. How does numpy.random.seed() enforce deterministic pseudorandom number generation in scientific experiments?

**Technical Breakdown**: np.random.seed(s) initializes the legacy Mersenne Twister (MT19937) pseudo-random number generator (PRNG) state space (2^19937 - 1). When fixed, subsequent PRNG state transitions produce identical sequences of deterministic floating-point values. In CPython, state mutation is handled at the C-extension layer (_multiarray_umath). In modern NumPy, np.random.default_rng(seed) is preferred for isolated BitGenerator instances (PCG64), but np.random.seed remains the standard global state locker for reproducibility across CI/CD and academic baselines.


In [1]:
import numpy as np
# seed: integer, optional
np.random.seed(0) # Set the random seed for reproducibility
# loc: mean of the distribution. float or array_like of floats
# scale: standard deviation of the distribution. Must be non-negative.
# size: number of samples to generate, int or tuple of ints(m, n, k), optional 
run_a = np.random.normal(loc=50, scale=10, size=3)

np.random.seed(0)
run_b = np.random.normal(loc=50, scale=10, size=3)

print("Run A:", run_a)
print("Run B:", run_b)
print("State Match Verified:", np.array_equal(run_a, run_b))


Run A: [67.64052346 54.00157208 59.78737984]
Run B: [67.64052346 54.00157208 59.78737984]
State Match Verified: True


### 2. What are the runtime mechanics and memory layout of numpy.random.normal(loc, scale, size)?

**Technical Breakdown**: Generates draws from a Gaussian distribution with probability density function. $$f(x) = \frac{1}{\sigma \sqrt{2\pi}}\exp\left(-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2\right)$$ Internally, CPython invokes the Box-Muller transform or the Ziggurat algorithm inside NumPy's compiled C code. It allocates an immutable contiguous block of IEEE 754 double-precision floats (float64, 8 bytes per element) in C-major (row-major) order. Memory complexity is $\mathcal{O}(N)$ where N = prod(size), and time complexity is $\mathcal{O}(N)$.


In [2]:
import numpy as np

data = np.random.normal(loc=50.0, scale=10.0, size=1000)

print("Type:", type(data))
print("Dtype:", data.dtype)
print("Shape:", data.shape)
print("Total Bytes Allocated:", data.nbytes)
print("First 3 elements:", data[:3])


Type: <class 'numpy.ndarray'>
Dtype: float64
Shape: (1000,)
Total Bytes Allocated: 8000
First 3 elements: [72.40893199 68.6755799  40.2272212 ]


### 3. How does numpy.mean() evaluate arithmetic averages and how does it handle accumulation overflow?

**Technical Breakdown**: `numpy.mean()` calculates the arithmetic mean by summing all elements and dividing the total by the number of elements.

$$
\bar{x} = \frac{1}{N}\sum_{i=1}^{N}x_i
$$

Where:
- $\bar{x}$ = Arithmetic mean.
- $N$ = Total number of observations.
- $x_i$ = Individual observation.
- $\sum$ = Summation of all observations.

Conceptually, NumPy performs two operations(Internal Computation):
1. **Accumulation:** Calculates the sum of the input elements.
2. **Division:** Divides the accumulated sum by the number of elements. 

It operates over contiguous memory buffers using SIMD vectorized addition (e.g., AVX-512/AVX2). For float arrays, intermediate summation preserves precision via pairwise summation or double-precision accumulators to mitigate floating-point rounding errors. Time complexity is $\mathcal{O}(N)$, and auxiliary space complexity is $\mathcal{O}(1)$.


In [3]:
import numpy as np

data = np.array([10.5, 20.5, 30.0, 40.0], dtype=np.float64)
mean_val = np.mean(data)

print("Calculated Mean:", mean_val)
print("Return Type:", type(mean_val))


Calculated Mean: 25.25
Return Type: <class 'numpy.float64'>


### 4. What is the algorithmic complexity and sorting behavior of numpy.median()?

**Technical Breakdown**: np.median() computes the 50th percentile. If array length N is odd, it returns the middle element; if even, it returns the mean of the two middle elements. Unlike Python's native statistics.median which requires a full $\mathcal{O}(N \log(N))$ TimSort, numpy.median() uses the Introselect algorithm (a hybrid of Quickselect and Heapsort), achieving an average time complexity of $\mathcal{O}(N)$ with worst-case $\mathcal{O}(N \log(N))$.


In [4]:
import numpy as np

odd_arr = np.array([7.0, 1.0, 3.0])
even_arr = np.array([7.0, 1.0, 3.0, 9.0])

print("Odd Median:", np.median(odd_arr))
print("Even Median:", np.median(even_arr))


Odd Median: 3.0
Even Median: 5.0


### 5. How does scipy.stats.mode() behave across SciPy versions and how are multimodal outputs structured?

**Technical Breakdown**: Computes the modal (most frequent) value. Historically, scipy.stats.mode returned a ModeResult(mode=array, count=array). In SciPy >= 1.9.0, keepdims=True or keepdims=False controls dimension retention. On continuous floating-point arrays generated by PRNGs, each value is typically unique, resulting in the smallest value being returned with a frequency count of 1.


In [5]:
import numpy as np
from scipy import stats

discrete_data = np.array([1, 2, 2, 3, 3, 3, 4])
mode_res = stats.mode(discrete_data, keepdims=False) # Change True/False and see the difference in output
print("Mode Result Object:", mode_res)
print("Modal Element:", mode_res.mode)
print("Modal Count:", mode_res.count)


Mode Result Object: ModeResult(mode=np.int64(3), count=np.int64(3))
Modal Element: 3
Modal Count: 3


### 6. What are the statistical assumptions and return types of scipy.stats.ttest_1samp()?

**Technical Breakdown**: Evaluates the null hypothesis $H_0$ against $H_1$ using Student's t-statistic, 
$$
\begin{aligned}
H_0 : \mu = \mu_0 \\
H_1 : \mu \neq \mu_0 \\
t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}
\end{aligned}
$$
where s is the sample standard deviation with n - 1 degrees of freedom (ddof=1). Returns a TtestResult named tuple containing statistic and pvalue (two-tailed by default). Time complexity is $\mathcal{O}(N)$ to evaluate sample mean and sample variance.

Understanding the Results:
- T-statistic: Measures how far the sample mean is from the hypothesized population mean in standard-error units.
- P-value(Probability Value): Measures how unusual the observed result (or a more extreme one) would be if the null hypothesis were true.
- Degrees of freedom: For a one-sample t-test, calculated as: `df = n - 1`
- Confidence interval: Provides a range of plausible values for the difference between the population mean and the hypothesized mean.

In [6]:
import numpy as np
from scipy import stats

np.random.seed(0)
sample = np.random.normal(loc=50, scale=10, size=1000)
# popmean: expected value in null hypothesis. float or array_like of floats. population mean to compare the sample against.
# ttest_1samp: test for the null hypothesis that the expected value (mean) of a sample of independent observations = the given population mean.
result = stats.ttest_1samp(sample, popmean=50.0)

print("T-Statistic:", result.statistic)
print("P-Value:", result.pvalue)
print("Degrees of Freedom:", result.df)
print("Confidence Interval:", result.confidence_interval())
print("Null Rejected at Alpha 0.05?:", result.pvalue < 0.05)


T-Statistic: -1.4492187904420446
P-Value: 0.14759031545459106
Degrees of Freedom: 999
Confidence Interval: ConfidenceInterval(low=np.float64(48.93462612126609), high=np.float64(50.160239728929994))
Null Rejected at Alpha 0.05?: False


### 7. What memory transformation occurs during .reshape(-1, 1) on a NumPy array?

**Technical Breakdown**: Reshaping converts a 1D array of shape (N,) into a 2D column vector of shape (N, 1). When the original array is C-contiguous, .reshape() modifies the array's shape and strides metadata without allocating new memory for the underlying buffer, returning a view ($\mathcal{O}(1)$ time and auxiliary space). The -1 argument instructs NumPy to dynamically deduce the dimension along that axis based on total array size.


In [7]:
import numpy as np

arr = np.arange(5, dtype=np.int32)
reshaped = arr.reshape(-1, 1)
# The strides attribute of a NumPy array indicates the number of bytes to step in each dimension when traversing the array. 
# It is a tuple that specifies how many bytes you need to move in memory to go from one element to the next along each axis. 
# The shape attribute indicates the dimensions of the array, and 
# shares_memory: checks if two arrays share the same memory buffer.
print("Original Strides:", arr.strides, "Shape:", arr.shape)
print("Reshaped Strides:", reshaped.strides, "Shape:", reshaped.shape)
print("Shares Memory Buffer:", np.shares_memory(arr, reshaped))


Original Strides: (4,) Shape: (5,)
Reshaped Strides: (4, 4) Shape: (5, 1)
Shares Memory Buffer: True


### 8. How does sklearn.preprocessing.StandardScaler calculate transformations and maintain state?

**Technical Breakdown:** `StandardScaler` standardizes each feature (column) by centering it around zero mean and scaling it to unit variance using the formula:

$$
z = \frac{x-\mu}{\sigma}
$$

- **`.fit(X)`**: Computes the mean (`mean_`) and variance (`var_`) of each feature (column) from the training data. It uses population variance (`ddof=0`).
- **`.scale_`**: Stores the scaling factor for each feature, calculated as the square root of its variance (`np.sqrt(var_)`). For zero-variance features, it uses a scaling factor of 1 to avoid division by zero.
- **`.transform(X)`**: Uses the learned means and scaling factors to standardize the input data.
- **`.fit_transform(X)`**: Performs both fitting and transformation in one step.

The output is a 2D NumPy array containing standardized values. Its dtype is generally `float64` for integer or `float64` input, but it can preserve `float32` when the input is `float32`.

**Important:** Fit the scaler only on the training data, then use the same fitted scaler to transform the test data. This prevents data leakage.


In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler

data = np.array([[10.0], [20.0], [30.0], [40.0], [50.0]])
scaler = StandardScaler()
scaled = scaler.fit_transform(data)

print("Fitted Mean:", scaler.mean_)
print("Fitted Scale (Std):", scaler.scale_)
print("Scaled Output:\n", scaled)


Fitted Mean: [30.]
Fitted Scale (Std): [14.14213562]
Scaled Output:
 [[-1.41421356]
 [-0.70710678]
 [ 0.        ]
 [ 0.70710678]
 [ 1.41421356]]


### 9. What is the execution mechanism and probability model of numpy.random.rand()?

**Technical Breakdown**: np.random.rand(d0, d1, ...) generates uniform continuous random floating-point samples over the half-open interval [0.0, 1.0). Multiplying by a scalar c invokes vectorized NumPy broadcasting, executing element-wise multiplication at the SIMD register level without Python-level iteration.


In [9]:
import numpy as np

np.random.seed(42)
X = np.random.rand(5, 1) * 10

print("Scaled Uniform Array:\n", X)
print("Min Value:", np.min(X), "| Max Value:", np.max(X))


Scaled Uniform Array:
 [[3.74540119]
 [9.50714306]
 [7.31993942]
 [5.98658484]
 [1.5601864 ]]
Min Value: 1.5601864044243652 | Max Value: 9.50714306409916


### 10. How does sklearn.preprocessing.PolynomialFeatures(degree=2) build interaction design matrices?

**Technical Breakdown**: Transforms an input design matrix X in R^{N x D} into an expanded feature space containing all polynomial combinations of degree <= d. For D=1 and degree=2, it produces columns [1, x, x^2] (if include_bias=True). Combinatorially, number of output features is binom(D + d, d). Output is an allocated dense 2D float array.
**Technical Breakdown:**

`sklearn.preprocessing.PolynomialFeatures(degree=2)` generates new features by creating polynomial combinations of the original input features up to degree 2.

For example, given two input, one input features:

$$
X = [x_0, x_1] \qquad X = [x_0]
$$

The respective transformed features are:

$$
[1,\ x_0,\ x_1,\ x_0^2,\ x_0x_1,\ x_1^2] \qquad [1,\ x_0,\ x_0^2]
$$

- **`degree=2`**: Generates polynomial terms up to degree 2, including squared terms and pairwise interaction terms.
- **`include_bias=True`** (default): Adds a constant bias column of 1s.
- **`.fit(X)`**: Determines the number of input features and the resulting polynomial feature combinations. It does not calculate statistical parameters such as mean or variance.
- **`.transform(X)`**: Generates the polynomial features using the learned feature combinations.
- **`.fit_transform(X)`**: Performs fitting and transformation in one step.
- **`.get_feature_names_out()`**: Returns the names of the generated features.


In [10]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[2.0, 3.0], [4.0, 5.0]]) # [[x0, x1]]
poly = PolynomialFeatures(degree=2, include_bias=True) 
X_poly = poly.fit_transform(X)

print("Feature Names:", poly.get_feature_names_out())
print("Transformed Matrix:\n", X_poly)


Feature Names: ['1' 'x0' 'x1' 'x0^2' 'x0 x1' 'x1^2']
Transformed Matrix:
 [[ 1.  2.  3.  4.  6.  9.]
 [ 1.  4.  5. 16. 20. 25.]]


### 11. What are the memory and index structures when initializing a pandas.DataFrame from an unshaped NumPy array?

**Technical Breakdown**: pd.DataFrame(data, columns=['Value']) wraps the contiguous 1D NumPy ndarray buffer. It constructs a RangeIndex(start=0, stop=N, step=1) for rows and an Index(['Value'], dtype='object') for columns. If input data is a 1D ndarray, Pandas instantiates a single 1D Block within its underlying BlockManager (or PyArrow/ArrayManager backend).


In [11]:
import numpy as np
import pandas as pd

raw = np.array([12.3, 45.6, 78.9, np.nan])
df = pd.DataFrame(raw, columns=['Value'])
print("DataFrame:",df, sep="\n")
print("DataFrame Info:")
print(df.info())
print("Underlying Array Reference:", df['Value'].to_numpy() is raw)


DataFrame:
   Value
0   12.3
1   45.6
2   78.9
3    NaN
DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Value   3 non-null      float64
dtypes: float64(1)
memory usage: 164.0 bytes
None
Underlying Array Reference: False


### 12. How does statsmodels.api.stats.DescrStatsW encapsulate descriptive statistics?

**Technical Breakdown**: DescrStatsW is a class for weighted and unweighted descriptive statistics of 1D and 2D numeric series. If weights=None, uniform weights ($w_i$ = 1) are applied. It evaluates summary statistics lazily or caches derived properties (mean, var, std_mean) upon initial access, avoiding redundant multi-pass traversals over large arrays.


In [12]:
import numpy as np
import statsmodels.api as sm

data = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
desc = sm.stats.DescrStatsW(data)

print("Class Type:", type(desc))
print("Computed Mean:", desc.mean)
print("Standard Error of Mean:", desc.std_mean)


Class Type: <class 'statsmodels.stats.weightstats.DescrStatsW'>
Computed Mean: 30.0
Standard Error of Mean: 7.0710678118654755


### 13. What degrees of freedom does DescrStatsW.var apply compared to NumPy's default variance?

**Technical Breakdown**: By default, DescrStatsW.var applies Bessel's correction (ddof=1), yielding unbiased sample variance $ s^2 = \frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar{x})^2 $. Conversely, np.var(data) defaults to ddof=0 (population variance $ \sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2 $). Failing to align degrees of freedom produces numerical discrepancies in downstream hypothesis testing.

* Population mean ($\mu$): Average of all values in the entire population.
* Sample mean ($\bar{x}$): Average of values from a subset (sample) of the population.



In [13]:
import numpy as np
import statsmodels.api as sm

arr = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
desc = sm.stats.DescrStatsW(arr)

print("DescrStatsW.var (ddof=1):", desc.var)
print("np.var default (ddof=0):", np.var(arr))
print("np.var matched (ddof=1):", np.var(arr, ddof=1))


DescrStatsW.var (ddof=1): 4.0
np.var default (ddof=0): 4.0
np.var matched (ddof=1): 4.571428571428571


### 14. How does numpy.percentile() evaluate quartiles and calculate the Interquartile Range (IQR)?

**Technical Breakdown**: The IQR is defined as Q3 - Q1 (75th percentile minus 25th percentile). np.percentile() uses linear interpolation between sorted array elements (default method='linear'). Sorting the array incurs $\mathcal{O}(N \log(N))$ complexity. The IQR serves as a non-parametric metric of statistical dispersion robust against extreme outliers.


In [14]:
import numpy as np

data = np.array([1, 2, 9, 12, 15, 18, 5, 6, 7, 19, 27])
q75 = np.percentile(data, 75)
q25 = np.percentile(data, 25)
q75_sorted = np.percentile(np.sort(data), 75)
q25_sorted = np.percentile(np.sort(data), 25)
iqr = q75 - q25
iqr_sorted = q75_sorted - q25_sorted

print(f"Q1: {q25}, Q3: {q75}, Calculated IQR: {iqr}")
print(f"Q1 (Sorted): {q25_sorted}, Q3 (Sorted): {q75_sorted}, Calculated IQR (Sorted): {iqr_sorted}")
# percentile automatically sorts the data before calculating the percentile, so the results should be the same whether you sort the data first or not.


Q1: 5.5, Q3: 16.5, Calculated IQR: 11.0
Q1 (Sorted): 5.5, Q3 (Sorted): 16.5, Calculated IQR (Sorted): 11.0


### 15. How does DescrStatsW.tconfint_mean() compute confidence intervals?

**Technical Breakdown**: Computes confidence limits using Student's t distribution: $\bar{x} \pm t_{\text{crit},\,\alpha/2,\,df}\times SE$, where:
* $\bar{x}$= Sample mean
* $t_{\text{crit}}$ = Critical t-value
* $\alpha$= Significance level
* df = N-1 = Degrees of freedom
* $SE = \frac{s}{\sqrt{N}}$ = Standard error of the sample mean

Default alpha=0.05 produces a 95% two-sided confidence interval. Returns a tuple (lower_bound, upper_bound) of IEEE floats.


In [15]:
import numpy as np
import statsmodels.api as sm

np.random.seed(42)
sample = np.random.normal(loc=100.0, scale=15.0, size=50)
desc = sm.stats.DescrStatsW(sample)
# alpha: significance level for the confidence interval. float, optional
# alternative: specifies the alternative hypothesis. {'two-sided', 'larger', 'smaller'}, optional
ci_low, ci_high = desc.tconfint_mean(alpha=0.05, alternative='two-sided')

print(f"95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"95% CI(Larger):{desc.tconfint_mean(alpha=0.05, alternative='larger')}")
print(f"95% CI(smaller):{desc.tconfint_mean(alpha=0.05, alternative='smaller')}")


95% CI: [92.6377, 100.5981]
95% CI(Larger):(np.float64(93.29729698339578), inf)
95% CI(smaller):(-inf, np.float64(99.93848585892003))


### 16. What occurs when sklearn.preprocessing.StandardScaler processes invariant (zero-variance) columns?

**Technical Breakdown**: When feature variance is zero (sigma = 0), division by standard deviation would yield IEEE 754 nan or inf via 0/0. StandardScaler prevents division-by-zero errors by setting scale factors equal to 1.0 (or leaving zeroed values untouched) when variance falls below machine epsilon, returning an output array populated by zeros.


In [16]:
import numpy as np
from sklearn.preprocessing import StandardScaler

flat_data = np.array([[5.0], [5.0], [5.0], [5.0]])
scaler = StandardScaler()
transformed = scaler.fit_transform(flat_data)

print("Scale attribute:", scaler.scale_)
print("Transformed Output (Zero Variance):\n", transformed)


Scale attribute: [1.]
Transformed Output (Zero Variance):
 [[0.]
 [0.]
 [0.]
 [0.]]


### 17. How does scipy.stats.ttest_1samp handle empty arrays or arrays containing fewer than 2 elements?

**Technical Breakdown**: Student's t-test requires degrees of freedom df = n - 1 >= 1 to calculate unbiased variance. When passed an empty array or single-element array, the sample variance computation evaluates to NaN (due to division by zero), causing ttest_1samp to output statistic=nan, pvalue=nan without throwing an unhandled exception.


In [17]:
import numpy as np
from scipy import stats

empty_arr = np.array([], dtype=np.float64)
single_arr = np.array([42.0])

res_empty = stats.ttest_1samp(empty_arr, 10.0)
res_single = stats.ttest_1samp(single_arr, 10.0)
print("Empty Input Test:", res_empty)
print("Single Input Test:", res_single)


Empty Input Test: TtestResult(statistic=np.float64(nan), pvalue=np.float64(nan), df=np.float64(nan))
Single Input Test: TtestResult(statistic=np.float64(nan), pvalue=np.float64(nan), df=np.int64(0))


C:\Users\jaich\AppData\Local\Temp\ipykernel_29996\2393948032.py:7: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  res_empty = stats.ttest_1samp(empty_arr, 10.0)
c:\Users\jaich\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:1227: RuntimeWarning: divide by zero encountered in divide
  var *= (n / (n-ddof))  # to avoid error on division by zero
c:\Users\jaich\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:1227: RuntimeWarning: invalid value encountered in multiply
  var *= (n / (n-ddof))  # to avoid error on division by zero


### 18. How does scipy.stats.mode resolve multimodal datasets when keepdims=True?

**Technical Breakdown**: When multiple values share the highest frequency count, scipy.stats.mode returns the smallest value among the ties. Specifying keepdims=True guarantees that the returned mode and count arrays preserve dimensionality identical to the input tensor rather than collapsing the reduced axis.


In [18]:
import numpy as np
from scipy import stats

bimodal_data = np.array([[10, 10, 20], [20, 5, 2]])
res = stats.mode(bimodal_data, keepdims=True,axis=1)
print("Mode Result Object:", res)

print("Resolved Mode (Smallest Tie):", res.mode[0])
print("Max Frequency Count:", res.count[0])


Mode Result Object: ModeResult(mode=array([[10],
       [ 2]]), count=array([[2],
       [1]]))
Resolved Mode (Smallest Tie): [10]
Max Frequency Count: [2]


### 19. How does setting include_bias=False in PolynomialFeatures alter output shapes?

**Technical Breakdown**: By default (include_bias=True), PolynomialFeatures prepends an intercept column containing all ones (x^0 = 1). Setting include_bias=False omits this column, reducing total feature count from binom(D+d, d) to binom(D+d, d) - 1, preventing collinearity issues with estimators that automatically fit an intercept term.


In [19]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[3.0], [5.0]])
poly_with_bias = PolynomialFeatures(degree=2, include_bias=True)
poly_no_bias = PolynomialFeatures(degree=2, include_bias=False)

out_with = poly_with_bias.fit_transform(X)
out_without = poly_no_bias.fit_transform(X)

print("With Bias Shape:", out_with.shape)
print("With Bias Output:\n", out_with)
print("Without Bias Shape:", out_without.shape)
print("Without Bias Output:\n", out_without)


With Bias Shape: (2, 3)
With Bias Output:
 [[ 1.  3.  9.]
 [ 1.  5. 25.]]
Without Bias Shape: (2, 2)
Without Bias Output:
 [[ 3.  9.]
 [ 5. 25.]]


### 20. What is the numerical impact of population variance (ddof=0) versus sample variance (ddof=1) in NumPy?

**Technical Breakdown**: Sample variance $s^2 = \frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar{x})^2$ is an unbiased estimator of population variance $\sigma^2$. The divisor N - 1 corrects for downward bias caused by evaluating sample deviations around the sample mean $\bar{x}$ rather than the true population mean $\mu$. NumPy defaults to ddof=0.

* Population variance (ddof=0): Divides the sum of squared deviations by N.
* Sample variance (ddof=1): Divides the sum of squared deviations by N-1 (Bessel's correction).

In [20]:
import numpy as np

sample = np.array([4.0, 8.0, 6.0, 5.0, 3.0])
var_pop = np.var(sample, ddof=0)
var_sample = np.var(sample, ddof=1)

print(f"Population Variance (N): {var_pop:.4f}")
print(f"Sample Variance (N-1): {var_sample:.4f}")
print(f"Correction Ratio: {var_sample / var_pop:.4f} vs {len(sample)/(len(sample)-1):.4f}")


Population Variance (N): 2.9600
Sample Variance (N-1): 3.7000
Correction Ratio: 1.2500 vs 1.2500


### 21. How are weighted descriptive calculations performed inside statsmodels.stats.weightstats.DescrStatsW?

**Technical Breakdown**: When initialized with weights vector w, DescrStatsW computes weighted mean and weighted sample variance (assuming frequency weights).
$$
 \bar{x}_w = \frac{\sum_{i=1}^{N} w_i x_i}{\sum_{i=1}^{N} w_i} \\[10pt]
 s_w^2 = \frac{\sum_{i=1}^{N} w_i(x_i-\bar{x}_w)^2}{\sum_{i=1}^{N} w_i-1}
$$
Time complexity remains $\mathcal{O}(N)$ with contiguous array operations.


In [21]:
import numpy as np
import statsmodels.api as sm

values = np.array([10.0, 20.0, 30.0])
weights = np.array([1.0, 1.0, 8.0])  # Heavily skew towards 30

desc_w = sm.stats.DescrStatsW(values, weights=weights)

print("Weighted Mean:", desc_w.mean)
print("Unweighted Arithmetic Mean:", np.mean(values))


Weighted Mean: 27.0
Unweighted Arithmetic Mean: 20.0


### 22. What runtime behavior occurs when evaluating np.median() on arrays containing np.nan values?

**Technical Breakdown**: np.median() propagates NaN values. If an array contains a single IEEE 754 NaN, comparison operations (<, >) fail during the partition/selection phase, causing np.median() to return np.nan. To ignore invalid entries, use np.nanmedian().


In [22]:
import numpy as np

dirty_arr = np.array([1.0, 5.0, np.nan, 10.0, 15.0, 20.0])
med_standard = np.median(dirty_arr)
med_nanrobust = np.nanmedian(dirty_arr)

print("Standard Median (Polluted):", med_standard)
print("Nan-Robust Median (Clean):", med_nanrobust)


Standard Median (Polluted): nan
Nan-Robust Median (Clean): 10.0


### 23. How does directional hypothesis testing work using scipy.stats.ttest_1samp's alternative argument?

**Technical Breakdown**: In SciPy >= 1.6.0, the parameter alternative accepts 'two-sided', 'less', or 'greater'. For alternative='greater', the test evaluates H1: mu > mu0. The p-value is determined by integrating the Student's t distribution tail corresponding to the directional inequality, halving or shifting the tail probability.


In [23]:
import numpy as np
from scipy import stats

np.random.seed(123)
data = np.random.normal(loc=52.0, scale=5.0, size=50)

two_sided = stats.ttest_1samp(data, 50.0, alternative='two-sided')
one_sided_g= stats.ttest_1samp(data, 50.0, alternative='greater')
one_sided_l= stats.ttest_1samp(data, 50.0, alternative='less')

print(f"Two-Sided P-Value: {two_sided.pvalue:.4f}")
print(f"One-Sided (Greater) P-Value: {one_sided_g.pvalue:.4f}")
print(f"One-Sided (Less) P-Value: {one_sided_l.pvalue:.4f}")


Two-Sided P-Value: 0.0187
One-Sided (Greater) P-Value: 0.0094
One-Sided (Less) P-Value: 0.9906


### 24. What are the constraints and behavior of StandardScaler when given a single sample vector?

**Technical Breakdown**: When given a shape (1, D), StandardScaler.fit() computes variance across zero degrees of freedom (N=1). The variance evaluates to 0.0, and the scaler sets scale_ to 1.0 (avoiding division by zero), resulting in a transformed output matrix populated by 0.0.


In [24]:
import numpy as np
from sklearn.preprocessing import StandardScaler

single_sample = np.array([[42.0]])
scaler = StandardScaler()
out = scaler.fit_transform(single_sample)

print("Fitted Mean:", scaler.mean_)
print("Scale Factor:", scaler.scale_)
print("Transformed Result:\n", out)


Fitted Mean: [42.]
Scale Factor: [1.]
Transformed Result:
 [[0.]]


### 25. How does PolynomialFeatures(interaction_only=True) prevent self-multiplied exponential terms?

**Technical Breakdown**: When interaction_only=True, features of the form $x_i^p$ where p > 1 are excluded. The transformer generates solely cross-product interactions $x_i * x_j$ for i != j alongside degree-1 terms. This curbs dimensionality expansion when modeling collaborative interaction effects without assuming non-linear self-power scaling.


In [25]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[2.0, 3.0]])
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_inter = poly.fit_transform(X)

print("Transformed Features:", poly.get_feature_names_out())
print("Values Matrix:\n", X_inter)


Transformed Features: ['x0' 'x1' 'x0 x1']
Values Matrix:
 [[2. 3. 6.]]


### 26. How do alpha levels adjust confidence bounds in DescrStatsW.tconfint_mean()?

**Technical Breakdown**: The critical value $t_{crit}$ scales inversely with $\alpha$. Lowering $\alpha$ from 0.05 (95% CI) to 0.01 (99% CI) shifts the distribution critical value outwards into the distribution tails, widening the interval estimate to ensure higher statistical coverage probability.


In [26]:
import numpy as np
import statsmodels.api as sm

np.random.seed(99)
data = np.random.normal(loc=25.0, scale=4.0, size=30)
desc = sm.stats.DescrStatsW(data)

ci_95 = desc.tconfint_mean(alpha=0.05)
ci_99 = desc.tconfint_mean(alpha=0.01)

print(f"95% Bounds Width: {ci_95[1] - ci_95[0]:.4f}")
print(f"99% Bounds Width: {ci_99[1] - ci_99[0]:.4f}")


95% Bounds Width: 2.5308
99% Bounds Width: 3.4108


### 27. How is Tukey's Interquartile Range (IQR) rule applied to detect statistical outliers?

**Technical Breakdown**: Under Tukey's fences, an observation x is flagged as an outlier if x < Q1 - 1.5 * IQR or x > Q3 + 1.5 * IQR. In Python, this is executed by computing percentile scalars and evaluating a boolean vector mask over the NumPy array ($\mathcal{O}(N)$ filter complexity).


In [27]:
import numpy as np

data = np.array([10.0, 12.0, 11.0, 13.0, 12.0, 100.0, -50.0])
q25, q75 = np.percentile(data, [25, 75])
iqr = q75 - q25
print(f"Q1: {q25}, Q3: {q75}, IQR: {iqr}")
lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr

outliers = data[(data < lower_bound) | (data > upper_bound)]
print(f"Lower Fence: {lower_bound}, Upper Fence: {upper_bound}")
print("Detected Outliers:", outliers)


Q1: 10.5, Q3: 12.5, IQR: 2.0
Lower Fence: 7.5, Upper Fence: 15.5
Detected Outliers: [100. -50.]


### 28. How does scipy.stats.ttest_1samp handle zero-variance arrays, and what exceptions must be trapped?

**Technical Breakdown**: If an input array consists of identical elements (e.g., [5.0, 5.0, 5.0]), sample standard deviation is s = 0. Division by zero inside the t-statistic equation ($t = \frac{\bar{x} - \mu_0}{0}$) results in RuntimeWarning: invalid value encountered in divide, producing statistic=nan, pvalue=nan without throwing a native Python exception. Production pipelines must trap this using numpy.isnan() checks.


In [28]:
import numpy as np
from scipy import stats
import warnings

arr = np.array([5.0, 5.0, 5.0, 5.0])

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    res = stats.ttest_1samp(arr, popmean=5.0)
    print("Test Result:", res)
    print("Warning Trapped:", len(w) > 0 and w[-1].category.__name__)


Test Result: TtestResult(statistic=np.float64(nan), pvalue=np.float64(nan), df=np.int64(3))
Warning Trapped: RuntimeWarning
